# Module 21 — Send, subgraphs, streaming, and the reducer trap

**THE ONE IDEA:** four features that only make sense once the loop is a graph — and one
trap that silently eats your data.

| | |
|---|---|
| **`Send`** | fan out to N copies of a node, one per item, decided **at runtime** |
| **subgraph** | a graph used as a node, with its **own** state schema |
| **`stream_mode`** | `values` · `updates` · `messages` · `debug` |
| **the trap** | two parallel branches writing one key **without a reducer** → one wins, silently |

`Send` is where module 06's sectioning becomes orchestrator-workers: **the model can
decide how many workers to spawn.** That is the workflow/agent boundary, in code.

No API key — deterministic, free.


In [ ]:
import sys; sys.path.insert(0, "..")   # shared _providers.py / _tools.py at the phase root
import operator
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send
from _tools import run_tool

class S(TypedDict):
    topics:  list
    findings: Annotated[list, operator.add]     # reducer: parallel writes ACCUMULATE
    count:    int                               # NO reducer — watch this one

## `Send` — fan out, decided at runtime

`fan_out` returns a **list of `Send` objects**. The number of workers is computed from
state, not written into the graph. Three topics now, seven tomorrow, no rewiring.

In [ ]:
def fan_out(state: S):
    return [Send("worker", {"topics": [t], "findings": [], "count": 0})
            for t in state["topics"]]

def worker(state: S):
    topic = state["topics"][0]
    return {"findings": [f"{topic}: {run_tool('search_policy', {'query': topic})[:46]}"],
            "count": 1}                          # every worker writes count=1

def gather(state: S):
    print(f"  gathered {len(state['findings'])} findings, count={state['count']}")
    return {}

b = StateGraph(S)
b.add_node("worker", worker); b.add_node("gather", gather)
b.add_conditional_edges(START, fan_out, ["worker"])
b.add_edge("worker", "gather"); b.add_edge("gather", END)
graph = b.compile()
print("graph compiled — three topics will fan out to three workers")

## The reducer-collision trap

Three workers each returned `count: 1`. `findings` has a reducer, so all three
accumulated. `count` has none — so what happens?

**In LangGraph 1.2 it raises `InvalidUpdateError` and refuses to run.** Older versions
took last-write-wins *silently*, which is what `8.agents/04_langgraph_deep.md` still
describes. The framework got stricter. Run it and see.

In [ ]:
from langgraph.errors import InvalidUpdateError
try:
    out = graph.invoke({"topics": ["ltv", "erc", "deposit"], "findings": [], "count": 0})
    print("no error — this build takes last-write-wins")
    print(f"  findings={len(out['findings'])}  count={out['count']} (expected 3)")
except InvalidUpdateError as e:
    print("InvalidUpdateError, as designed:\n ", str(e).splitlines()[0])
    print("\n-> the graph REFUSED to run rather than lose a write. Good default.")

# The fix: give the contended key a reducer too.
class S2(TypedDict):
    topics:   list
    findings: Annotated[list, operator.add]
    count:    Annotated[int, operator.add]        # <- the one-word fix

b2 = StateGraph(S2)
b2.add_node("worker", worker); b2.add_node("gather", gather)
b2.add_conditional_edges(START, fan_out, ["worker"])
b2.add_edge("worker", "gather"); b2.add_edge("gather", END)
graph2 = b2.compile()
out = graph2.invoke({"topics": ["ltv", "erc", "deposit"], "findings": [], "count": 0})
print(f"\nwith a reducer on count -> findings={len(out['findings'])}  count={out['count']}  CORRECT")
for f in out["findings"]: print("   ", f)

## Subgraphs — a graph as a node

A subgraph has its **own state schema**. The outer state is not automatically visible,
so you pass in only what it needs and map the result back.

In [ ]:
class Sub(TypedDict):
    query: str
    result: str

def sub_lookup(s: Sub): return {"result": run_tool("search_policy", {"query": s["query"]})}
def sub_trim(s: Sub):   return {"result": s["result"][:40] + "..."}

sb = StateGraph(Sub)
sb.add_node("lookup", sub_lookup); sb.add_node("trim", sub_trim)
sb.add_edge(START, "lookup"); sb.add_edge("lookup", "trim"); sb.add_edge("trim", END)
sub = sb.compile()

def call_sub(state: S):
    r = sub.invoke({"query": state["topics"][0], "result": ""})   # explicit hand-off
    return {"findings": [r["result"]]}

ob = StateGraph(S); ob.add_node("sub", call_sub)
ob.add_edge(START, "sub"); ob.add_edge("sub", END)
outer = ob.compile()
print(outer.invoke({"topics": ["rates"], "findings": [], "count": 0})["findings"])

## The four stream modes

In [ ]:
init = {"topics": ["ltv", "erc"], "findings": [], "count": 0}

print("stream_mode='updates'  — only the DIFF each node produced (most efficient):")
for u in graph2.stream(dict(init), stream_mode="updates"):
    for node, ch in u.items(): print(f"   {node:7} -> {ch}")

print("\nstream_mode='values'   — the FULL state after each step (for a UI):")
for v in graph2.stream(dict(init), stream_mode="values"):
    print(f"   findings={len(v.get('findings', []))} count={v.get('count')}")

print("""
LESSON - four features, one trap.

  Send        the NUMBER of workers is computed from state at runtime. Module 06
              fixed its three sections in a Python list; this decides per input.
              Let the MODEL produce that list and you have orchestrator-workers -
              the exact point a workflow becomes an agent.

  subgraph    its own state schema. Nothing leaks in or out implicitly - you map
              both directions by hand. That isolation is the feature: a worker
              cannot accidentally read the orchestrator's context.

  stream      'updates' for machines (just the diff), 'values' for a UI (full
              state), 'messages' for token-by-token inside a node, 'debug' for
              internal events. Pass a list to subscribe to several.

  THE TRAP    findings accumulated. count collided. In LangGraph 1.2 that is a
              hard InvalidUpdateError; in older builds it was SILENT last-write-
              wins, which is what 8.agents/04_langgraph_deep.md still says. Either
              way the rule holds: if a key CAN be written by more than one branch,
              it needs Annotated[T, reducer]. 'Can' means could-ever, not usually.
""")

---

**Next:** Block H — `../H_memory/22_memory_short_term_strategies.ipynb`